# Lab 03 — Event Hubs Silver

Read the Event Hubs Bronze Delta stream, parse Wikimedia JSON, clean and validate important fields, and deduplicate events before writing them to the Silver Delta table.


## 1. Load the shared configuration


In [0]:
%run ./lab03_config


## 2. Define the Wikimedia event schema

The schema contains the fields used by this lab. The complete original JSON remains available in Bronze.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType
)


wikimedia_meta_schema = StructType([
    StructField("id", StringType(), True),
    StructField("dt", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("stream", StringType(), True),
    StructField("topic", StringType(), True),
    StructField("uri", StringType(), True),
    StructField("request_id", StringType(), True),
])

wikimedia_event_schema = StructType([
    StructField("$schema", StringType(), True),
    StructField("meta", wikimedia_meta_schema, True),
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("namespace", LongType(), True),
    StructField("title", StringType(), True),
    StructField("title_url", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("patrolled", BooleanType(), True),
    StructField("server_name", StringType(), True),
    StructField("server_script_path", StringType(), True),
    StructField("server_url", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("_producer_id", StringType(), True),
    StructField("_source_system", StringType(), True),
    StructField("_ingested_at_utc", StringType(), True),
])

print("Wikimedia schema created.")


## 3. Read the Bronze table as a stream


In [0]:
bronze_stream_df = (
    spark.readStream
    .table(eventhub_bronze_table)
)

print(f"Reading Bronze table: {eventhub_bronze_table}")


## 4. Parse and clean the JSON payload


In [0]:
parsed_df = (
    bronze_stream_df
    .withColumn(
        "event",
        F.from_json(
            F.col("raw_event"),
            wikimedia_event_schema
        )
    )
)

cleaned_df = (
    parsed_df
    .select(
        F.col("event.meta.id").alias("event_id"),
        F.to_timestamp(
            F.col("event.meta.dt")
        ).alias("event_timestamp"),
        F.col("event.meta.domain").alias("domain"),
        F.col("event.meta.stream").alias("source_stream"),
        F.col("event.id").alias("revision_id"),
        F.trim(F.col("event.type")).alias("change_type"),
        F.col("event.namespace").alias("wiki_namespace"),
        F.trim(F.col("event.title")).alias("title"),
        F.col("event.title_url").alias("title_url"),
        F.trim(F.col("event.comment")).alias("comment"),
        F.col("event.timestamp").alias("event_epoch_seconds"),
        F.trim(F.col("event.user")).alias("user_name"),
        F.coalesce(
            F.col("event.bot"),
            F.lit(False)
        ).alias("is_bot"),
        F.coalesce(
            F.col("event.minor"),
            F.lit(False)
        ).alias("is_minor"),
        F.col("event.patrolled").alias("is_patrolled"),
        F.trim(F.col("event.wiki")).alias("wiki"),
        F.trim(
            F.col("event._producer_id")
        ).alias("producer_id"),
        F.trim(
            F.col("event._source_system")
        ).alias("source_system"),
        F.to_timestamp(
            F.col("event._ingested_at_utc")
        ).alias("producer_ingested_at"),
        F.col("eventhub_name"),
        F.col("eventhub_partition"),
        F.col("eventhub_offset"),
        F.col("eventhub_enqueued_at"),
        F.col("bronze_ingested_at"),
        F.col("raw_event")
    )
    .withColumn(
        "event_timestamp",
        F.coalesce(
            F.col("event_timestamp"),
            F.to_timestamp(
                F.from_unixtime("event_epoch_seconds")
            )
        )
    )
    .withColumn(
        "title",
        F.when(
            F.length("title") > 0,
            F.col("title")
        )
    )
    .withColumn(
        "user_name",
        F.when(
            F.length("user_name") > 0,
            F.col("user_name")
        )
    )
)

print("Cleaned Silver streaming DataFrame prepared.")
cleaned_df.printSchema()


## 5. Validate and deduplicate

Valid events must have an event ID, timestamp, change type, wiki, and title. A watermark limits how long Spark keeps deduplication state.


In [0]:
valid_silver_df = (
    cleaned_df
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("event_timestamp").isNotNull())
    .filter(F.col("change_type").isNotNull())
    .filter(F.col("wiki").isNotNull())
    .filter(F.col("title").isNotNull())
    .filter(
        F.col("source_system")
        == F.lit("wikimedia_recentchange")
    )
    .withWatermark(
        "event_timestamp",
        "1 day"
    )
    .dropDuplicates(["event_id"])
)

print("Silver validation and deduplication rules applied.")


## 6. Write the Silver Delta stream

`availableNow` processes all currently unprocessed Bronze data and then stops. Re-running the notebook processes only new rows because the checkpoint is preserved.


In [0]:
silver_query = (
    valid_silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        eventhub_silver_checkpoint_path
    )
    .queryName("lab03_eventhub_silver_processing")
    .trigger(availableNow=True)
    .toTable(eventhub_silver_table)
)

silver_query.awaitTermination()

print("Silver processing completed.")
print(f"Target table: {eventhub_silver_table}")
print(f"Checkpoint: {eventhub_silver_checkpoint_path}")


## 7. Verify Silver quality and uniqueness


In [0]:
silver_df = spark.table(eventhub_silver_table)

quality_result = (
    silver_df
    .agg(
        F.count("*").alias("silver_rows"),
        F.countDistinct("event_id").alias(
            "distinct_event_ids"
        ),
        F.sum(
            F.col("event_id").isNull().cast("int")
        ).alias("null_event_ids"),
        F.sum(
            F.col("event_timestamp").isNull().cast("int")
        ).alias("null_event_timestamps"),
        F.sum(
            F.col("title").isNull().cast("int")
        ).alias("null_titles")
    )
)

display(quality_result)


In [0]:
display(
    silver_df
    .orderBy(F.col("event_timestamp").desc())
    .limit(20)
)


## 8. Optional invalid-event analysis

This batch query shows why Bronze events would fail the Silver rules.


In [0]:
invalid_event_summary = (
    spark.table(eventhub_bronze_table)
    .withColumn(
        "event",
        F.from_json(
            F.col("raw_event"),
            wikimedia_event_schema
        )
    )
    .select(
        F.when(
            F.col("event").isNull(),
            "invalid_json"
        )
        .when(
            F.col("event.meta.id").isNull(),
            "missing_event_id"
        )
        .when(
            F.coalesce(
                F.to_timestamp(F.col("event.meta.dt")),
                F.to_timestamp(
                    F.from_unixtime(
                        F.col("event.timestamp")
                    )
                )
            ).isNull(),
            "missing_event_timestamp"
        )
        .when(
            F.trim(F.col("event.type")).isNull(),
            "missing_change_type"
        )
        .when(
            F.trim(F.col("event.wiki")).isNull(),
            "missing_wiki"
        )
        .when(
            F.trim(F.col("event.title")).isNull(),
            "missing_title"
        )
        .when(
            F.trim(F.col("event._source_system"))
            != "wikimedia_recentchange",
            "unexpected_source"
        )
        .otherwise("valid")
        .alias("validation_result")
    )
    .groupBy("validation_result")
    .count()
    .orderBy(F.col("count").desc())
)

display(invalid_event_summary)


In [0]:
bronze_count = spark.table(
    eventhub_bronze_table
).count()

silver_count = spark.table(
    eventhub_silver_table
).count()

print(f"Bronze row count: {bronze_count}")
print(f"Silver row count: {silver_count}")